# 팀 공통 모델 평가 Notebook

각 팀원은 **자기 학습 모델/LoRA Adapter는 그대로 사용**하고,
같은 평가 데이터와 같은 생성/평가 조건으로 점수만 다시 계산합니다.

공통 조건:
- `validation_canonical_981.csv`
- 동일 System Prompt
- `do_sample=False`
- `max_new_tokens=300`
- Multi-reference: `reference_responses_json`
- 동일 평가 함수

In [ ]:
import gc, json, time, re
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
from sentence_transformers import SentenceTransformer
from rouge_score import rouge_scorer
from sacrebleu.metrics import CHRF

print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# ============================================================
# ★ 팀원별로 여기만 수정
# ============================================================

MODEL_NAME = "TEAM_MEMBER_MODEL"

MODEL_ID = "LGAI-EXAONE/EXAONE-3.5-7.8B-Instruct"
MODEL_REVISION = "0ff6b5ec7c13b049b253a16a889aa269e6b79a94"

ADAPTER_DIR = Path(r"C:\project\outputs\YOUR_ADAPTER_FOLDER\adapter")
CANONICAL_CSV = Path(r"C:\project\data\validation_canonical_981.csv")

OUTPUT_DIR = Path(r"C:\project\outputs\common_evaluation") / MODEL_NAME
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MAX_NEW_TOKENS = 300

assert ADAPTER_DIR.exists(), f"Adapter 폴더가 없습니다: {ADAPTER_DIR}"
assert CANONICAL_CSV.exists(), f"평가 CSV가 없습니다: {CANONICAL_CSV}"

print("MODEL_NAME:", MODEL_NAME)
print("ADAPTER_DIR:", ADAPTER_DIR)
print("CANONICAL_CSV:", CANONICAL_CSV)

In [ ]:
SYSTEM_PROMPT = '''
당신은 쇼핑몰 고객센터 상담 AI입니다.

고객 문의와 문의 유형(Intent)을 참고하여
고객에게 전달할 최종 답변을 작성하세요.

규칙:
- 한국어 존댓말을 사용하세요.
- 답변은 간결하고 명확하게 작성하세요.
- 제공되지 않은 정책, 금액, 기간, 사실을 임의로 만들지 마세요.
- 불필요한 분석 과정은 출력하지 마세요.
- 고객에게 전달할 최종 답변만 작성하세요.
'''.strip()

def make_user_text(intent, question):
    return f"문의 유형: {intent}\n고객 문의: {question}"

eval_df = pd.read_csv(CANONICAL_CSV)

required_cols = ["의도","문의 내용","응답","reference_responses_json"]
missing = [c for c in required_cols if c not in eval_df.columns]
assert not missing, f"필수 컬럼 누락: {missing}"
assert len(eval_df) == 981, f"평가 데이터가 981개가 아닙니다: {len(eval_df)}"

print("✅ 평가 데이터:", eval_df.shape)

In [ ]:
gc.collect()
torch.cuda.empty_cache()

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    revision=MODEL_REVISION,
    trust_remote_code=True,
    use_fast=True,
)

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "left"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    revision=MODEL_REVISION,
    trust_remote_code=True,
    quantization_config=bnb_config,
    device_map={"": 0},
    torch_dtype=torch.bfloat16,
    low_cpu_mem_usage=True,
)

model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)
model.eval()
model.config.use_cache = True

print("✅ 모델 로드 완료")

In [ ]:
rows = []
start_all = time.perf_counter()

for i, (_, row) in enumerate(eval_df.iterrows(), start=1):
    intent = str(row["의도"])
    question = str(row["문의 내용"])

    messages = [
        {"role":"system","content":SYSTEM_PROMPT},
        {"role":"user","content":make_user_text(intent, question)},
    ]

    prompt_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(
        prompt_text,
        return_tensors="pt",
        add_special_tokens=False,
    ).to("cuda")

    t0 = time.perf_counter()

    with torch.no_grad():
        generated = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    seconds = time.perf_counter() - t0

    generated_tokens = generated[0, inputs["input_ids"].shape[1]:]
    answer = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

    rows.append({
        "index": i-1,
        "의도": intent,
        "문의 내용": question,
        "응답": row["응답"],
        "reference_responses_json": row["reference_responses_json"],
        "generated_response": answer,
        "generation_seconds": seconds,
        "generated_tokens": len(generated_tokens),
    })

    if i % 50 == 0 or i == len(eval_df):
        print(f"{i:>4} / {len(eval_df)} 완료")

pred_df = pd.DataFrame(rows)

PREDICTION_CSV = OUTPUT_DIR / "predictions.csv"
pred_df.to_csv(PREDICTION_CSV, index=False, encoding="utf-8-sig")

print("\n✅ 생성 완료")
print("총 시간:", round((time.perf_counter()-start_all)/60, 2), "분")
print("빈 답변:", pred_df["generated_response"].str.strip().eq("").sum())
print("저장:", PREDICTION_CSV)

In [ ]:
rouge = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=False)
chrf_metric = CHRF()

def extract_key_facts(text):
    text = str(text)
    facts = set()
    facts.update(re.findall(r"\{\{[^{}]+\}\}", text))
    facts.update(re.findall(r"\[[^\[\]]+\]", text))
    facts.update(re.findall(r"\d+(?:[.,]\d+)?", text))
    facts.update(re.findall(r"\d{2,4}-\d{3,4}-\d{4}", text))
    facts.update(re.findall(r"https?://[^\s]+", text))
    return facts

def key_fact_f1(prediction, reference):
    pred_facts = extract_key_facts(prediction)
    ref_facts = extract_key_facts(reference)

    if not pred_facts and not ref_facts:
        return np.nan
    if not pred_facts or not ref_facts:
        return 0.0

    common = pred_facts & ref_facts
    precision = len(common) / len(pred_facts)
    recall = len(common) / len(ref_facts)

    if precision + recall == 0:
        return 0.0

    return 2 * precision * recall / (precision + recall)

def key_fact_f1_best(prediction, references):
    scores = [key_fact_f1(prediction, ref) for ref in references]
    valid_scores = [x for x in scores if not np.isnan(x)]
    return np.nan if not valid_scores else max(valid_scores)

def rouge_l_best(prediction, references):
    return max(rouge.score(ref, prediction)["rougeL"].fmeasure for ref in references)

def chrf_best(prediction, references):
    return max(chrf_metric.sentence_score(prediction, [ref]).score / 100 for ref in references)

def has_bad_repetition(text):
    text = str(text).strip()
    if not text:
        return True

    sentences = [x.strip() for x in re.split(r"[.!?]\s*", text) if x.strip()]
    if len(sentences) < 2:
        return False

    counts = {}
    for sentence in sentences:
        counts[sentence] = counts.get(sentence, 0) + 1

    return max(counts.values()) >= 3

def is_korean_response(text):
    return len(re.findall(r"[가-힣]", str(text))) >= 5

def is_clean_answer(text):
    text = str(text).strip()
    if not text:
        return False

    bad_patterns = ["<think>", "</think>", "assistant:", "system:", "user:"]
    lower_text = text.lower()

    return not any(pattern in lower_text for pattern in bad_patterns)

print("✅ 공통 평가 함수 준비 완료")

In [ ]:
del model
del base_model
gc.collect()
torch.cuda.empty_cache()

SEMANTIC_MODEL_ID = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

semantic_model = SentenceTransformer(
    SEMANTIC_MODEL_ID,
    device="cuda",
)

def semantic_scores_multi(predictions, refs_list):
    pred_embeddings = semantic_model.encode(
        [str(x) for x in predictions],
        batch_size=64,
        convert_to_tensor=True,
        normalize_embeddings=True,
        show_progress_bar=True,
    )

    flat_refs = []
    spans = []
    cursor = 0

    for refs in refs_list:
        refs = [str(x) for x in refs]
        start = cursor
        flat_refs.extend(refs)
        cursor += len(refs)
        spans.append((start, cursor))

    ref_embeddings = semantic_model.encode(
        flat_refs,
        batch_size=64,
        convert_to_tensor=True,
        normalize_embeddings=True,
        show_progress_bar=True,
    )

    scores = []

    for i, (start, end) in enumerate(spans):
        sims = pred_embeddings[i] @ ref_embeddings[start:end].T
        scores.append(float(sims.max().item()))

    return scores

refs_list = [json.loads(x) for x in pred_df["reference_responses_json"]]

pred_df["semantic_similarity"] = semantic_scores_multi(
    pred_df["generated_response"].tolist(),
    refs_list,
)

print("✅ Semantic:", round(pred_df["semantic_similarity"].mean()*100, 2), "%")

In [ ]:
pred_df["key_fact_f1"] = [
    key_fact_f1_best(pred, refs)
    for pred, refs in zip(pred_df["generated_response"], refs_list)
]

pred_df["rougeL"] = [
    rouge_l_best(pred, refs)
    for pred, refs in zip(pred_df["generated_response"], refs_list)
]

pred_df["chrf"] = [
    chrf_best(pred, refs)
    for pred, refs in zip(pred_df["generated_response"], refs_list)
]

pred_df["repetition_free"] = [
    not has_bad_repetition(x)
    for x in pred_df["generated_response"]
]

pred_df["korean_response"] = [
    is_korean_response(x)
    for x in pred_df["generated_response"]
]

pred_df["clean_answer"] = [
    is_clean_answer(x)
    for x in pred_df["generated_response"]
]

key_valid = pred_df["key_fact_f1"].dropna()

semantic = pred_df["semantic_similarity"].mean()
key_f1 = key_valid.mean()
rouge_l = pred_df["rougeL"].mean()
chrf = pred_df["chrf"].mean()
repetition = pred_df["repetition_free"].mean()
korean = pred_df["korean_response"].mean()
clean = pred_df["clean_answer"].mean()

cs_quality_score = (
    semantic * 55
    + rouge_l * 20
    + chrf * 15
    + repetition * 5
    + korean * 5
)

print("\n=== 공통 평가 결과 ===")
print(f"Semantic Similarity : {semantic*100:.2f}%")
print(f"Key Fact F1         : {key_f1*100:.2f}% ({len(key_valid)}/{len(pred_df)} 평가 가능)")
print(f"ROUGE-L             : {rouge_l*100:.2f}%")
print(f"chrF                : {chrf*100:.2f}%")
print(f"Repetition Free     : {repetition*100:.2f}%")
print(f"Korean Response     : {korean*100:.2f}%")
print(f"Clean Answer        : {clean*100:.2f}%")
print(f"CS Quality Score    : {cs_quality_score:.2f}/100")

In [ ]:
DETAIL_PATH = OUTPUT_DIR / "common_evaluation_detail.csv"
SUMMARY_PATH = OUTPUT_DIR / "common_evaluation_summary.csv"

pred_df.to_csv(
    DETAIL_PATH,
    index=False,
    encoding="utf-8-sig",
)

summary_df = pd.DataFrame([{
    "model_name": MODEL_NAME,
    "semantic_similarity_percent": semantic * 100,
    "key_fact_f1_percent": key_f1 * 100,
    "key_fact_valid_count": len(key_valid),
    "rougeL_percent": rouge_l * 100,
    "chrf_percent": chrf * 100,
    "repetition_free_percent": repetition * 100,
    "korean_response_percent": korean * 100,
    "clean_answer_percent": clean * 100,
    "cs_quality_score_100": cs_quality_score,
    "avg_generation_seconds": pred_df["generation_seconds"].mean(),
    "median_generation_seconds": pred_df["generation_seconds"].median(),
    "p95_generation_seconds": pred_df["generation_seconds"].quantile(0.95),
    "empty_response_count": pred_df["generated_response"].str.strip().eq("").sum(),
}])

summary_df.to_csv(
    SUMMARY_PATH,
    index=False,
    encoding="utf-8-sig",
)

display(summary_df.round(4))

print("\n✅ 저장 완료")
print("상세:", DETAIL_PATH)
print("요약:", SUMMARY_PATH)

## 팀원 사용법

1. 학습은 다시 하지 않습니다.
2. 각자 학습한 LoRA Adapter를 그대로 둡니다.
3. 2번 셀에서 `MODEL_NAME`, `ADAPTER_DIR`만 자기 것으로 수정합니다.
4. 팀 공통 `validation_canonical_981.csv`를 사용합니다.
5. `Run All` 합니다.
6. 마지막 `common_evaluation_summary.csv`를 팀 공용 폴더에 제출합니다.

이렇게 해야 **학습 모델만 다르고, 생성 조건과 평가 기준은 동일**해집니다.